# Landmark re-analysis (castrate-sensitive) and drug-holiday fraction vs TTP (all cohorts)

Branch `corrected-fitting-protocol`. Code: `analysis/holiday_fraction_utils.py`, `analysis/landmark_holiday_analysis.py`
(fits and statistics reuse `patients/castrate_sensitive.py`, `analysis/castrate_sensitive_stats.py`,
`patients/first_cycle_fitting.py`). Patient data are not tracked in the public repository.

**Definitions, fixed before any association was computed**
* **First cycle** = first drug-on row -> the row at which treatment restarts (start of cycle 2). Bruchovsky: first `CycleId == 2` row;
  castrate-sensitive: first `Cycle == 2` row; Zhang (no cycle label): first off -> on transition after treatment has started.
  13 of 17 Zhang files begin with pre-treatment, off-drug rows, so for them the cycle starts at the first drug-on row, not the first record.
* **Off (holiday) fraction** = time with no drug / time in window, step-holding the drug state of the preceding row. Windows: first cycle;
  whole record (Bruchovsky, Zhang only); up to TTP (Bruchovsky, Zhang only). The castrate-sensitive workbook contains the first cycle only, so
  for that cohort the 'full treatment history' fraction is not available and equals the first-cycle fraction at most.
* **TTP / event**: Bruchovsky, Zhang as in the SI right-censoring notebooks (TTP = last time PSA < 1.2 x first PSA; event = max PSA > 1.2 x first PSA).
  Castrate-sensitive: TTE and status from `progression_outcomes.tsv` (TSV) and the workbook-reconciled version (see `SI_Castrate_Sensitive_Cohort.ipynb`).
* **Drug state for the castrate-sensitive holiday fraction**: 'union' (any of ADT / abiraterone / ARSI) is the physically meaningful holiday;
  'ADT-only' is the state used for fitting. Both are reported.
* **Assumption that cannot be checked from the files**: the TSV/Summary times share their origin with the sheet's `Days` (first ADT dose).
* All tests are exploratory and not corrected for multiplicity.

In [1]:
import os, sys, warnings, numpy as np, pandas as pd, matplotlib
warnings.filterwarnings("ignore"); matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
ROOT = os.path.abspath(".."); os.chdir(ROOT)
for sub in ("utils", "patients", "analysis"): sys.path.insert(0, os.path.join(ROOT, sub))
import castrate_sensitive as cs, castrate_sensitive_stats as st
import holiday_fraction_utils as hf, landmark_holiday_analysis as lh
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
tabs = cs.load_tabs(); summ = cs.load_summary(); outc = cs.load_outcomes()
as_supplied, reconciled, audit = cs.reconcile_outcomes(outc, summ)
OUTS = {"TSV": as_supplied, "Reconciled": reconciled}

## 1. Castrate-sensitive cohort: time from the end of the first cycle to progression

In [2]:
B = hf.holiday_table("Bruchovsky"); Z = hf.holiday_table("Zhang")
CSu = hf.cs_table(as_supplied, tabs, "union"); CSa = hf.cs_table(as_supplied, tabs, "ADT")
L = CSu.set_index("PatientId").cycle1_length
tab = pd.DataFrame({"first_cycle_length_d": L, "TTE_TSV": as_supplied.TTE, "event_TSV": as_supplied.Event,
                    "TTE_reconciled": reconciled.TTE, "event_reconciled": reconciled.Event})
tab["TTE_TSV_minus_L"] = tab.TTE_TSV - tab.first_cycle_length_d; tab["TTE_reconciled_minus_L"] = tab.TTE_reconciled - tab.first_cycle_length_d
print(tab.astype(int).to_string()); print("\nsmallest TTE - L:", int(tab[["TTE_TSV_minus_L","TTE_reconciled_minus_L"]].min().min()), "d (nobody is excluded by the shift)")
FITV = lh.fit_all_variants(tabs)   # 8 outcome-blind fit variants (origin x drug state x priors), ~45 s

           first_cycle_length_d  TTE_TSV  event_TSV  TTE_reconciled  event_reconciled  TTE_TSV_minus_L  TTE_reconciled_minus_L
PatientId                                                                                                                     
102                         253      840          1             462                 1              587                     209
103                         640     1987          0            1987                 0             1347                    1347
104                         211     1933          0            1933                 0             1722                    1722
105                         227     1963          0            1963                 0             1736                    1736
106                         162     1296          1            1296                 1             1134                    1134
107                         219      849          1             849                 1              630         

In [3]:
prim = lh.landmark_primary(FITV, OUTS, L)
cols = ["outcomes","time","set","biomarker","N","events","HR","lo","hi","p_wald","C","p_C_one","p_C_two","rho_T","p_rho"]
print(prim[cols].round(3).to_string(index=False))

  outcomes                 time set biomarker  N  events    HR    lo     hi  p_wald     C  p_C_one  p_C_two  rho_T  p_rho
       TSV                  raw   A  AT score 11       3 0.029 0.000 19.864   0.288 0.778    0.089    0.162  0.364  0.272
       TSV                  raw   A      eTTP 11       3 0.035 0.000 16.959   0.288 0.778    0.089    0.162  0.364  0.272
       TSV                  raw   B  AT score 12       3 0.028 0.000 19.702   0.285 0.786    0.082    0.154  0.217  0.499
       TSV                  raw   B      eTTP 12       3 0.023 0.000 24.407   0.288 0.786    0.082    0.156  0.112  0.729
       TSV                  raw   C  AT score 16       5 0.925 0.372  2.302   0.867 0.540    0.399    0.818  0.199  0.461
       TSV                  raw   C      eTTP 16       5 0.914 0.358  2.330   0.850 0.548    0.378    0.758  0.110  0.684
       TSV minus cycle-1 length   A  AT score 11       3 0.029 0.000 19.864   0.288 0.778    0.089    0.162  0.382  0.247
       TSV minus cycle-1

In [4]:
G = lh.landmark_grid(FITV, OUTS, L)
GS = G.groupby("time").apply(lh.grid_summary).T; print(GS.round(3).to_string())
m = G[G.HR.notna()].pivot_table(index=["origin","drug","priors","outcomes","set","biomarker"], columns="time", values=["C","HR","p_C_one"])
dC = (m["C"]["adjusted"] - m["C"]["raw"]).abs().max(); dH = np.log(m["HR"]["adjusted"] / m["HR"]["raw"]).abs().max()
cross = m[(m["p_C_one"]["raw"] < .05) != (m["p_C_one"]["adjusted"] < .05)]["p_C_one"]
print(f"max |change in C| = {dC:.3f}; max |change in log HR| = {dH:.3f}; cells crossing one-sided P=.05 on shifting: {len(cross)}"); print(cross.round(3))
print("events for 80% power at HR 0.5 / 0.7 per SD:", round(lh.schoenfeld_events(.5),1), round(lh.schoenfeld_events(.7),1))
prim.to_csv("analysis/castrate_sensitive_landmark_primary.csv", index=False); G.to_csv("analysis/castrate_sensitive_landmark_grid.csv", index=False)

time                                     adjusted     raw
cells                                      96.000  96.000
estimable                                  92.000  92.000
HR_below_1                                 90.000  90.000
wald_p_below_05                             0.000   0.000
min_wald_p                                  0.081   0.088
permC_onesided_p_below_05                  26.000  25.000
permC_onesided_p_below_05_nondegenerate     4.000   3.000
setC_permC_p_below_05                       2.000   1.000
setC_cells                                 32.000  32.000
max |change in C| = 0.012; max |change in log HR| = 0.059; cells crossing one-sided P=.05 on shifting: 1
time                                                     adjusted    raw
origin    drug  priors           outcomes set biomarker                 
first_psa union Bruchovsky-style TSV      C   eTTP          0.049  0.054
events for 80% power at HR 0.5 / 0.7 per SD: 16.3 61.7


**Result.** Subtracting the first-cycle length does not change the conclusion. In the pre-declared primary specification (set C, all 16 patients,
5 events with the TSV outcomes) the AT-score hazard ratio per +1 SD moves from 0.93 to
0.89
(P = 0.87 -> 0.80); the eTTP hazard ratio from
0.91 to 0.87.
Across the 96-cell specification grid (92 estimable) no Wald P falls below .05 either before (minimum 0.088) or after
(minimum 0.081) the shift; the largest change in Harrell's C in any cell is 0.012.
One-sided permutation P < .05 occurs in 25 (raw) and 26 (shifted) cells, but almost all are the quasi-separated 3-event
sets A/B (HR ~ 0.03, CI spanning 0-20); in the full cohort (set C) it is 1 -> 2 of 32 cells, about what chance gives.
Why the shift changes so little: first-cycle length is unrelated to the AT score (see section 2) and only weakly related to TTE, so removing it re-ranks few event times.
With 5-6 events this cohort cannot show significance for any plausible effect size: ~16 events would be needed for 80% power at HR 0.5 per SD.

## 2. Is the first-cycle length coupled to the AT score or the holiday fraction?

In [5]:
prim_scored = st.score_fits(FITV[("treatment","ADT","Zhang-style")], cost=0.01).set_index("PatientId")
at_rank = pd.Series(st._rank_inf(prim_scored.AT_score.values), index=prim_scored.index)
for lab, s in [("TTE (TSV)", as_supplied.TTE), ("fitted turnover", prim_scored.turnover), ("AT score", at_rank),
               ("first-cycle off fraction, union", CSu.set_index("PatientId").off_cycle1), ("first-cycle off fraction, ADT-only", CSa.set_index("PatientId").off_cycle1)]:
    r, p = spearmanr(L, s.reindex(L.index)); print(f"  L vs {lab:36s} rho={r:+.3f}  P={p:.3f}")

  L vs TTE (TSV)                            rho=+0.471  P=0.066
  L vs fitted turnover                      rho=+0.009  P=0.974
  L vs AT score                             rho=+0.044  P=0.870
  L vs first-cycle off fraction, union      rho=+0.924  P=0.000
  L vs first-cycle off fraction, ADT-only   rho=+0.974  P=0.000


## 3. TTP vs drug-holiday fraction in the three cohorts

In [6]:
subB = set(pd.read_csv("analysis/holiday_endpoint_bruchovsky.csv").PatientId)      # the 40 analysed in the AT-score study
CST = {"union": CSu, "ADT-only": CSa}
print("Bruchovsky: first drug-on at t=0 in", int((B.t_first_on == 0).sum()), "/72; CycleId restart == drug-transition restart in", int(np.isclose(B.cycle1_end, B.restart_from_drug).sum()), "/72")
print("Zhang: files starting off drug:", int(Z.starts_off_drug.sum()), "/", len(Z), "; TTP = 0 (excluded):", sorted(Z.PatientId[Z.TTP <= 0]))
print("median off fraction  Bruchovsky first cycle / whole: %.3f / %.3f ;  Zhang: %.3f / %.3f ; castrate-sensitive first cycle (union / ADT-only): %.3f / %.3f" % (
      B.off_cycle1.median(), B.off_whole.median(), Z.off_cycle1.median(), Z.off_whole.median(), CSu.off_cycle1.median(), CSa.off_cycle1.median()))
HA = lh.holiday_associations(B, Z, CST, OUTS, subB)
print(HA.round(3).to_string(index=False))
HA.to_csv("analysis/holiday_fraction_associations.csv", index=False)
pd.concat([B.assign(cohort="Bruchovsky"), Z.assign(cohort="Zhang"), CSu.assign(cohort="Castrate-sensitive (union)"), CSa.assign(cohort="Castrate-sensitive (ADT-only)")]).to_csv("analysis/holiday_fraction_per_patient.csv", index=False)

Bruchovsky: first drug-on at t=0 in 72 /72; CycleId restart == drug-transition restart in 72 /72
Zhang: files starting off drug: 13 / 17 ; TTP = 0 (excluded): [1004, 1009]
median off fraction  Bruchovsky first cycle / whole: 0.569 / 0.487 ;  Zhang: 0.625 / 0.486 ; castrate-sensitive first cycle (union / ADT-only): 0.573 / 0.603
           dataset                                     subset     fraction              outcome  N  events    rho  p_rho    HR    lo    hi  p_wald     C  p_C_two
        Bruchovsky                                     all 72  first cycle                  TTP 72      21  0.096  0.424 0.977 0.631 1.514   0.919 0.550    0.487
        Bruchovsky                                     all 72  first cycle TTP - end of cycle 1 71      21 -0.423  0.000 1.810 1.047 3.129   0.034 0.392    0.132
        Bruchovsky                                     all 72 whole record                  TTP 72      21  0.187  0.116 0.821 0.528 1.278   0.383 0.614    0.110
        Bruchovsky    

In [7]:
CP = lh.coupling_table(B, Z, CST, OUTS, subB); print(CP.round(3).to_string(index=False)); CP.to_csv("analysis/holiday_fraction_coupling.csv", index=False)

                                     set  N  events  rho_off_vs_L  rho_L_vs_TTP  rho_off_vs_TTP  partial_rho_off_vs_TTP_given_L  partial_p  cox_adj_HR_off  cox_adj_lo  cox_adj_hi  cox_adj_p
                       Bruchovsky all 72 72      21         0.951         0.051           0.096                           0.153      0.201           2.226       0.624       7.946      0.218
                  Bruchovsky 40 analysed 40      15         0.861        -0.017           0.101                           0.228      0.157           4.012       0.646      24.903      0.136
                    Zhang all TTP>0 (15) 15      15         0.029         0.257           0.204                           0.203      0.468           0.736       0.411       1.318      0.302
          Castrate-sensitive union / TSV 16       5         0.924         0.471           0.453                           0.054      0.842           0.236       0.017       3.248      0.280
   Castrate-sensitive union / Reconciled 16       

## 4. Sensitivity: where the Zhang AT-score fits are made (published window vs first treated cycle)

In [8]:
ZFITS, ZSENS = lh.zhang_window_sensitivity(Z)
print("patients with <3 points in the fit window:", ZFITS.groupby(["window","priors"]).n_pts.apply(lambda s: int((s < 3).sum())).to_dict())
print(ZSENS.pivot_table(index=["priors","subset","endpoint"], columns="window", values=["N","rho","p"]).round(3).to_string())
ZFITS.to_csv("analysis/zhang_window_fits.csv", index=False); ZSENS.to_csv("analysis/zhang_window_sensitivity.csv", index=False)

patients with <3 points in the fit window: {('published', 'Bruchovsky-style (fR 1e-5, cost 0.1)'): 9, ('published', 'Zhang-style (fR 1e-2, cost 0.01)'): 9, ('treated', 'Bruchovsky-style (fR 1e-5, cost 0.1)'): 0, ('treated', 'Zhang-style (fR 1e-2, cost 0.01)'): 0}
                                                                                                               N                 p               rho        
window                                                                                                 published treated published treated published treated
priors                               subset              endpoint                                                                                           
Bruchovsky-style (fR 1e-5, cost 0.1) all with TTP>0      TTP                                                15.0    15.0     0.959   0.308     0.014  -0.282
                                                         first-cycle off fraction (same window as fit)      

## 5. Figure

In [9]:
exec(open("paper_figures/landmark_holiday_figure.py").read())

![landmark and holiday fraction](../analysis/landmark_holiday_fraction.png)

## Summary

**Holiday fraction vs TTP (first cycle and whole record; TTP as defined above).**
* Bruchovsky (72 patients): first-cycle off fraction rho = +0.10 (P = 0.42), Cox HR 0.98 per SD (0.63-1.51), C = 0.55, N = 72, events = 21; whole record rho = +0.19 (P = 0.12), Cox HR 0.82 per SD (0.53-1.28), C = 0.61, N = 72, events = 21.
* Zhang (15 patients with TTP > 0): first cycle rho = +0.20 (P = 0.47), Cox HR 0.74 per SD (0.41-1.33), C = 0.59, N = 15, events = 15; whole record rho = +0.43 (P = 0.11), Cox HR 0.50 per SD (0.25-1.03), C = 0.67, N = 15, events = 15.
* Castrate-sensitive (16; first cycle only): union rho = +0.45 (P = 0.08), Cox HR 0.28 per SD (0.07-1.17), C = 0.74, N = 16, events = 5; ADT-only rho = +0.52 (P = 0.04), Cox HR 0.32 per SD (0.09-1.14), C = 0.74, N = 16, events = 5 (TSV outcomes; reconciled outcomes in the table above).
* None of the Cox or concordance tests reaches P < .05 for the raw TTP. The only nominal Spearman P < .05 are the castrate-sensitive ADT-only cells, and these are explained by cycle length:
  the first-cycle off fraction is almost a monotone function of first-cycle length there (rho = +0.92 union, +0.97 ADT-only), and the partial correlation given cycle length is
  +0.05 (union) and +0.32 (ADT-only), P = 0.84 and 0.22. The same is true in Bruchovsky (rho off vs L = +0.95).
* Subtracting the end-of-cycle time from TTP creates an artefactual negative association in Bruchovsky (rho -0.42) because the off fraction rises with cycle length; do not interpret those rows as biology.
  In Zhang the whole-record fraction against TTP minus end of cycle 1 gives rho = +0.68 (P = 0.016, N = 12 because three patients have TTP before the cycle ends);
  one nominal result among 26 tests, in a cohort whose first-record PSA is pre-treatment for 13/17 patients, so it is not treated as evidence.
* Whole-record and up-to-TTP fractions share their window with TTP (more follow-up -> more cycles -> a more stable fraction), so a positive association there is partly built in.

**Zhang fit window (new finding).** 13 of 17 Zhang files begin with pre-treatment, off-drug rows. `TruncateDataframe` cuts at the first off -> on switch, which for those patients is the start of abiraterone, so the 'first cycle' that was
fitted is the pre-treatment segment (9 patients with < 3 points, i.e. the 'sparsity' seen earlier). Re-fitting on the first treated cycle (all 17 have >= 3 points; Zhang-style priors, first-measurement normalisation, kappa = 0.2) gives:
* AT score vs TTP: published truncation window (otherwise harmonised settings) rho = +0.37 (P = 0.173, N = 15) -> treated cycle rho = +0.09 (P = 0.752, N = 15); below the pole rho = +0.35 (P = 0.239, N = 13).
* AT score vs whole-record off fraction: published truncation window rho = +0.03 (P = 0.919, N = 15) -> treated cycle rho = +0.56 (P = 0.031, N = 15); below the pole rho = +0.72 (P = 0.006, N = 13).
* With Bruchovsky-style priors the treated-window AT score vs whole-record off fraction (below the pole) is rho = +0.70 (P = 0.011, N = 12); AT score vs TTP is not significant under either prior.

Interpretation: on a correct treated-cycle window the Zhang TTP association with the AT score is not supported, whereas the AT score does track the fraction of the history spent off drug in both Bruchovsky and Zhang.
This is a post hoc check on 15 patients (one main specification plus one prior sensitivity), prompted by the discovery of the window problem, so it is a hypothesis to confirm, not a result. The Bruchovsky holiday-fraction association comes from the earlier analysis in this branch (`analysis/holiday_endpoint.py`). The first-cycle off fraction column in `zhang_window_sensitivity.csv` shares the fit window and is therefore not an independent endpoint.